# Image Reconstruction

In [ ]:
# This cells setups the environment when executed in Google Colab.
try:
    import google.colab
    !curl -s https://raw.githubusercontent.com/ibs-lab/cedalion/dev/scripts/colab_setup.py -o colab_setup.py
    # Select branch with --branch "branch name" (default is "dev")
    %run colab_setup.py
except ImportError:
    pass

In [ ]:
%load_ext autoreload
%autoreload 2

## Notebook configuration 
Decide for an example dataset with a sparse probe or a high-density probe for DOT. 
The notebook will load example data accordingly.

Also specify, if precomputed results of the photon propagation should be used and
if the 3D visualizations should be interactive.

In [ ]:
DATASET = "fingertappingDOT" # high-density montage
HEAD_MODEL = "colin27"
FORWARD_MODEL = "MCX" # photon monte carlo
PRECOMPUTED_FLUENCE = True

# set this flag to True to enable interactive 3D plots
INTERACTIVE_PLOTS = False

In [ ]:
import pyvista as pv

if INTERACTIVE_PLOTS:
    pv.set_jupyter_backend("server")
else:
    pv.set_jupyter_backend("static")

import itertools
import time
import traceback
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as p
import numpy as np
import xarray as xr
from IPython.display import Image

import cedalion
import cedalion.dataclasses as cdc
import cedalion.datasets

# from cedalion.imagereco.solver import pseudo_inverse_stacked
import cedalion.dot
import cedalion.geometry.registration
import cedalion.geometry.segmentation
import cedalion.io
import cedalion.plots
import cedalion.sigproc.motion_correct as motion_correct
import cedalion.typing as cdt
import cedalion.vis.plot_sensitivity_matrix
import cedalion.xrutils as xrutils
from cedalion import units
from cedalion.io.forward_model import FluenceFile, load_Adot
from cedalion.sigproc.quality import measurement_variance
from cedalion.plots import image_recon_multi_view

# fail on unit errors
xrutils.unit_stripping_is_error()

xr.set_options(display_expand_data=False);

In [ ]:
# helper function to display gifs in rendered notbooks
def display_image(fname : str):
    display(Image(data=open(fname,'rb').read(), format='png'))

## Working Directory
In this notebook the output of the fluence and sensitivity calculations are stored in a temporary directory. This will be 
deleted when the notebook ends.

In [ ]:
temporary_directory = TemporaryDirectory()
tmp_dir_path = Path(temporary_directory.name)

## Load a finger-tapping dataset 

For this demo we load an example finger-tapping recording through either `cedalion.datasets.get_fingertapping` or `cedalion.datasets.get_fingertappingDOT`. 

The snirf files of these datasets contains a single NIRS element with one block of raw amplitude data.

In [ ]:
rec = cedalion.datasets.get_fingertappingDOT()

The location of the probes is obtained from the snirf metadata (i.e. /nirs0/probe/)

Note that units ('m') are adopted and the coordinate system is named 'digitized'.

In [ ]:
geo3d_meas = rec.geo3d
display(geo3d_meas)

The measurement list is a `pandas.DataFrame` that describes which source-detector pairs form channels.

In [ ]:
meas_list = rec._measurement_lists["amp"]
display(meas_list.head(5))

Event/stimulus information is also stored in a `pandas.DataFrame`. 

In [ ]:
rec.stim

For clarity, events are given more descriptive names:

In [ ]:
rec.stim.cd.rename_events(
    {
        "1": "Control",
        "2": "FTapping/Left",
        "3": "FTapping/Right",
        "4": "BallSqueezing/Left",
        "5": "BallSqueezing/Right",
    }
)

# count number of trials per trial_type
display(
    rec.stim.groupby("trial_type")[["onset"]]
    .count()
    .rename({"onset": "#trials"}, axis=1)
)

## Preprocessing
Perform motion correction, conversion to optical density and bandpass filtering.

In [ ]:
rec["od"] = cedalion.nirs.int2od(rec["amp"])
rec["od_tddr"] = motion_correct.tddr(rec["od"])
rec["od_wavelet"] = motion_correct.wavelet(rec["od_tddr"])

# bandpass filter the data
rec["od_freqfiltered"] = rec["od_wavelet"].cd.freq_filter(
    fmin=0.01, fmax=0.5, butter_order=4
)

## Calculate block averages in optical density

In [ ]:
# segment data into epochs
epochs = rec["od_freqfiltered"].cd.to_epochs(
    rec.stim,  # stimulus dataframe
    ["FTapping/Left", "FTapping/Right"],  # select fingertapping events, discard others
    before=5 * units.s,  # seconds before stimulus
    after=30 * units.s,  # seconds after stimulus
)

# calculate baseline
baseline = epochs.sel(reltime=(epochs.reltime < 0)).mean("reltime")

# subtract baseline
epochs_blcorrected = epochs - baseline

# group trials by trial_type. For each group individually average the epoch dimension
blockaverage = epochs_blcorrected.groupby("trial_type").mean("epoch")

## The TwoSurfaceHeadModel

The photon propagation considers the complete MRI scan, in which each voxel is attributed to one tissue type with its respective optical properties. However, the image reconstruction does not intend to reconstruct absorption changes in each voxel. The inverse problem is simplified, by considering only two surfaces (scalp and brain) and reconstruct only absorption changes in voxels close to these surfaces.

The class `cedalion.imagereco.forward_model.TwoSurfaceHeadModel` groups together the segmentation mask, landmark positions and affine transformations as well as the scalp and brain surfaces. The brain surface is calculated by grouping together white and gray matter masks. The scalp surface encloses the whole head.

In [ ]:
head = cedalion.dot.get_standard_headmodel(HEAD_MODEL)

In [ ]:
head

In [ ]:
head.landmarks

Changing between coordinate systems:

In [ ]:
head_ras = head.apply_transform(head.t_ijk2ras)
display(head_ras.crs)
display(head_ras.brain)

In [ ]:
head_ras

## Optode Registration
The optode coordinates from the recording must be aligned with the scalp surface. Currently, `cedaĺion` offers a simple registration method, which finds an affine transformation (scaling, rotating, translating) that matches the landmark positions of the head model and their digitized counter parts. Afterwards, optodes are snapped to the nearest vertex on the scalp.

In [ ]:
geo3d_snapped_ijk = head.align_and_snap_to_scalp(geo3d_meas)
display(geo3d_snapped_ijk)

## Simulate light propagation in tissue

`cedalion.imagereco.forward_model.ForwardModel` is a wrapper around pmcx. Using the data in the head model it prepares the inputs for either pmcx or NIRFASTer and offers functionality to calculate the sensitivty matrix.

In [ ]:
fwm = cedalion.dot.ForwardModel(head, geo3d_snapped_ijk, meas_list)

### Run the simulation

The `compute_fluence_mcx` and `compute_fluence_nirfaster` methods simulate a light source at each optode position and calculate the fluence in each voxel. By setting `RUN_PACKAGE`, you can choose between the pmcx or NIRFASTer package to perform this simulation.
PLEASE NOTE: if you USE_CACHED data (download the example data) be aware that the file is quite big (~2GB).

In [ ]:
if PRECOMPUTED_FLUENCE:
    if FORWARD_MODEL == "MCX":
        fluence_fname = cedalion.datasets.get_precomputed_fluence(DATASET, HEAD_MODEL)
    elif FORWARD_MODEL == "NIRFASTER":
        raise NotImplementedError(
            "Currently there are no precomputed NIRFASTER results available"
        )
else:
    fluence_fname = tmp_dir_path / "fluence.h5"

    if FORWARD_MODEL == "MCX":
        fwm.compute_fluence_mcx(fluence_fname)
    elif FORWARD_MODEL == "NIRFASTER":
        fwm.compute_fluence_nirfaster(fluence_fname)

The photon simulation yields the fluence in each voxel for each wavelength:

- `fluence_all` is a `xr.DataArray` with dimensions: ('label', 'wavelength', 'i', 'j', 'k'),

   i.e. for each optode and wavelength it stores the 3D image of the computed fluence in each voxel

- `fluence_at_optodes` is a `xr.DataArray` with dimensions: ('optode1', 'optode2', 'wavelength').

  It contains the fluence directly at the position of the optodes, used for normalization purposes.


Both arrays are stored on disk in the hdf5 file at `fluence_fname` and should be queried through `cedalion.io.forward_model.FluenceFile`.

Also, for a each combination of two optodes, the fluence in the voxels at the optode positions is calculated.

### Calculate the sensitivity matrices
The forward model's function `compute_sensitivity` calculates the sensitivity matrix from the fluence file and saves the result in a new file.

In [ ]:
if PRECOMPUTED_FLUENCE:
    Adot = cedalion.datasets.get_precomputed_sensitivity(DATASET, HEAD_MODEL)
else:
    sensitivity_fname = tmp_dir_path / "sensitivity.h5"
    fwm.compute_sensitivity(fluence_fname, sensitivity_fname)
    Adot = load_Adot(sensitivity_fname)

The sensitivity matrix describes how an absorption change at a given surface vertex changes the optical density in a given channel and wavelength. 

The coordinate `is_brain` holds a mask to distinguish brain and scalp voxels.

In [ ]:
# load and display sensitivity matrix
display(Adot)

## Reconstruct the Image

### Gaussian Spatial Basis Functions

After constructing the SBFs once, the prepared GaussianSpatialBasisFunctions object can be stored in an hdf5 file.
If the file already exists, load it.

In [ ]:
fname = tmp_dir_path / "sbf.h5"

if fname.exists():
    sbf = cedalion.dot.GaussianSpatialBasisFunctions.from_file(fname)
else:
    sbf = cedalion.dot.GaussianSpatialBasisFunctions(
        head_ras,
        Adot,
        mask_threshold=-2,
        threshold_brain=1 * units.mm,
        threshold_scalp=5 * units.mm,
        sigma_brain=1 * units.mm,
        sigma_scalp=5 * units.mm,
    )
    sbf.to_file(fname)


### Create combinations of ImageRecon parameters 

In [ ]:
timeseries = [
    # ('channel', 'wavelength', 'time')
    blockaverage.isel(reltime=[40], trial_type=0).rename({"reltime" : "time"}),
    # ('channel', 'wavelength')
    blockaverage.isel(reltime=40, trial_type=0),
    # ('trial_type', 'channel', 'wavelength', 'reltime')
    blockaverage.isel(reltime=[40], trial_type=[0]),
]

c_meas = measurement_variance(rec["od"].pint.dequantify(), calc_covariance=False)

sbfs = [sbf, None]
recon_modes = ["mua", "conc", "mua*mua2conc"]
brainonly_modes = [False, True]
alpha_meas_params = [0.01]
alpha_spatial_params = [None, 0.01]
c_meas_params = [None, c_meas]

combinations = list(
    itertools.product(
        timeseries,
        sbfs,
        recon_modes,
        brainonly_modes,
        alpha_meas_params,
        alpha_spatial_params,
        c_meas_params,
    )
)
print(f"generated {len(combinations)} combinations")

In [ ]:
def run_comb(i, catch):
    ts, sbf_, recon_mode, brainonly_mode, alpha_meas, alpha_spatial, c_meas_ = combinations[i]

    print(
        f"{i}/{len(combinations)}: {' x '.join(ts.dims)} sbf_={sbf_ is not None} "
        f"{recon_mode=} {brainonly_mode=} {alpha_meas=} {alpha_spatial=} "
        f"c_meas_={c_meas_ is not None}"
    )

    # construct ImageRecon object
    recon = cedalion.dot.ImageRecon(
        Adot,
        recon_mode=recon_mode,
        regularization_params=cedalion.dot.RegularizationParams(
            alpha_meas=alpha_meas,
            alpha_spatial=alpha_spatial,
            apply_c_meas=(c_meas_ is not None),
        ),
        spatial_basis_functions=sbf_,
        brain_only=brainonly_mode,
    )

    if catch:
        # run reconstruction, catch any exception
        try:
            recon_result = recon.reconstruct(ts, c_meas=c_meas_)
            print(f"recon_result dims:{recon_result.dims} shape: {recon_result.shape}")
        except Exception as ex:
            print("Error")
            print(traceback.format_exc())
            return None
        else:
            print("OK")
    else:
        # run reconstruction, don't catch any exception. jump into debugger
        recon_result = recon.reconstruct(ts, c_meas=c_meas)
        print(f"recon_result dims:{recon_result.dims} shape: {recon_result.shape}")

    return recon_result

### Run a single combination of recon settings. Entry point for debugging

In [ ]:
run_comb(16, catch=False)

### Run all combinations. Catch and count combinations that throw errors.

In [ ]:
print(80*"-")
oks = 0
for i in range(len(combinations)):
    oks += int(run_comb(i, catch=True) is not None)
    print(80*"-")

print("="*80)
print(f"OK={oks} / {len(combinations)}")
print("="*80)


## Inspect Reconstruction results

In [ ]:
plot_combinations = [
    i
    for i, (
        ts,
        sbf_,
        recon_mode,
        brainonly_mode,
        alpha_meas,
        alpha_spatial,
        c_meas_,
    ) in enumerate(combinations)
    if (recon_mode=="conc") and (brainonly_mode==False) and (ts.dims == ('trial_type', 'channel', 'wavelength', 'reltime'))
]
display(plot_combinations)

In [ ]:
for i_comb in plot_combinations:

    recon_result = run_comb(i_comb, catch=False)

    #display(recon_result)

    # reduce recon_result to vertex x chromo
    X_ts = recon_result.sel(vertex=recon_result.is_brain).isel(trial_type=0, reltime=0)

    image_recon_multi_view(
        X_ts,  # time series data; can be 2D (static) or 3D (dynamic)
        head,
        cmap="seismic",
        clim=(-1e-6, 1e-6),
        view_type="hbr_brain",
        title_str="HbO / uM",
        # filename=filename_multiview,
        SAVE=False,
        # time_range=(-5,30,0.5)*units.s,
        # fps=6,
        geo3d_plot=None,  #  geo3d_plot
        wdw_size=(1024, 768),
    )